In [7]:
def get_top_keywords(url: str, k: int = 10, return_details: bool = False):
    import re
    import urllib.request
    from urllib.parse import urlparse
    from collections import Counter
    from bs4 import BeautifulSoup, Comment
    from nltk.corpus import stopwords
    from nltk import wordpunct_tokenize
    import nltk
    import time

    # --------------------- ensure NLTK data ---------------------
    try:
        stopwords.words("english")
    except LookupError:
        nltk.download("stopwords")

    # --------------------- config ---------------------
    COMMON_NOISE_WORDS = set("""
    January debt est dec big than who use jun jan feb mar apr may jul august dec oct nov sep dec
    product continue one two three four five please thanks find helpful week job experience women girl
    apology read show eve knowledge benefit appointment street way staff salon discount gift cost thing
    world close party love letters rewards offers special close pack wed dollars voucher gifts vouchers
    welcome therefore march nights need name please show sisters thank menu today always time needs
    welcome march february april may june jully august september october november december day year
    month minute second seconds
    """.split())

    SPECIAL_CHARS_RE = re.compile(r"[^ \.\,\|\@#\$\%\^\&\*\(\)\_\+\=\-\[\]\{\}\;\:\'\"\\\<\>\?\/\.\,\-]")

    # --------------------- helpers ---------------------
    def _is_visible_text(element) -> bool:
        if element.parent.name in ["html", "style", "script", "head", "[document]", "img"]:
            return False
        if isinstance(element, Comment):
            return False
        return True

    def _extract_visible_text_from_html(html: bytes) -> str:
        soup = BeautifulSoup(html, "lxml")
        texts = soup.find_all(string=True)
        visible_texts = filter(_is_visible_text, texts)
        return " ".join(t.strip() for t in visible_texts)

    def _normalize_whitespace(text: str) -> str:
        lines = (line.strip() for line in text.splitlines())
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        return "\n".join(chunk for chunk in chunks if chunk)

    def _fetch_page(u: str, retries: int = 3):
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.5",
            "Accept-Encoding": "gzip, deflate",
            "Connection": "keep-alive",
            "Upgrade-Insecure-Requests": "1"
        }
        
        for attempt in range(retries):
            try:
                req = urllib.request.Request(u, headers=headers)
                with urllib.request.urlopen(req, timeout=20) as resp:
                    html = resp.read()
                soup = BeautifulSoup(html, "lxml")
                raw_text = _extract_visible_text_from_html(html)
                clean_text = _normalize_whitespace(raw_text)
                return clean_text, soup
            except urllib.error.HTTPError as e:
                if e.code == 403 and attempt < retries - 1:
                    # Wait before retrying on 403
                    time.sleep(2 ** attempt)
                    continue
                elif e.code == 403:
                    print(f"Warning: 403 Forbidden for {u}, returning empty result")
                    return "", BeautifulSoup("", "lxml")
                else:
                    raise
            except Exception as e:
                if attempt < retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    print(f"Warning: Failed to fetch {u}: {e}, returning empty result")
                    return "", BeautifulSoup("", "lxml")

    def _calculate_language_scores(text: str) -> dict:
        ratios = {}
        tokens = wordpunct_tokenize(text)
        words = [w.lower() for w in tokens]
        words_set = set(words)

        for lang in stopwords.fileids():
            try:
                stop_set = set(stopwords.words(lang))
            except Exception:
                continue
            common = words_set.intersection(stop_set)
            ratios[lang] = len(common)
        return ratios

    def _detect_language_and_stopwords(text: str):
        ratios = _calculate_language_scores(text)
        if not ratios:
            # fallback to English
            return "english", set(stopwords.words("english"))

        detected_lang = max(ratios, key=ratios.get)

        # some nltk ids (like 'porter') are not stopword sets; fall back to english if needed
        try:
            sw = set(stopwords.words(detected_lang))
        except Exception:
            detected_lang, sw = "english", set(stopwords.words("english"))
        return detected_lang, sw

    def _clean_text_to_words(text: str, stopword_list: set) -> list:
        words = []
        LETTERS_ONLY_RE = re.compile(r"[^a-zA-ZåäöÅÄÖ]+")
    
        for raw_word in text.split():
            token = LETTERS_ONLY_RE.sub("", raw_word).lower()
    
            if (
                len(token) > 1
                and not token[0].isdigit()
                and token not in stopword_list
                and token not in COMMON_NOISE_WORDS
                and not token.isdigit()
            ):
                words.append(token)
    
        return words


    def _split_url_host(u: str) -> list:
        parsed = urlparse(u)
        host = parsed.hostname or ""
        parts = []
        for chunk in host.split("."):
            chunk = chunk.lower()
            if chunk not in ["", "https", "www", "com", "-", "php", "pk", "fi", "http"]:
                parts.append(chunk)
        return parts

    def _split_url_path_and_query(u: str, host_parts: list) -> list:
        path_tokens = []
        for segment in u.split("/"):
            for dot_part in segment.split("."):
                for dash_part in dot_part.split("-"):
                    token = dash_part.lower()
                    if (
                        token
                        and token not in [
                            "https", "www", "com", "-", "php", "pk", "fi",
                            "https:", "http", "http:", "http:",
                        ]
                        and token not in host_parts
                    ):
                        path_tokens.append(token)
        return path_tokens

    def _extract_tag_texts(soup, tag_name: str) -> list:
        out = []
        for el in soup.find_all(tag_name):
            t = el.get_text(strip=True).lower()
            if t:
                out.append(t)
        return out

    def _explode_texts_to_words(text_list: list) -> list:
        out = []
        for text in text_list:
            for comma_chunk in text.split(","):
                for w in comma_chunk.split():
                    out.append(w)
        return out

    def _extract_headers_anchors_title_words(soup):
        h1 = _explode_texts_to_words(_extract_tag_texts(soup, "h1"))
        h2 = _explode_texts_to_words(_extract_tag_texts(soup, "h2"))
        h3 = _explode_texts_to_words(_extract_tag_texts(soup, "h3"))
        h4 = _explode_texts_to_words(_extract_tag_texts(soup, "h4"))
        h5 = _explode_texts_to_words(_extract_tag_texts(soup, "h5"))
        h6 = _explode_texts_to_words(_extract_tag_texts(soup, "h6"))
        a = _explode_texts_to_words(_extract_tag_texts(soup, "a"))
        ti = _explode_texts_to_words(_extract_tag_texts(soup, "title"))
        return h1, h2, h3, h4, h5, h6, a, ti

    def _tf_score(freq: int, total_tokens: int) -> float:
        # Keep your original behavior
        if total_tokens < 50:
            return (freq / 100.0) * 50
        else:
            return (freq / 100.0) * 20

    def _compute_keyword_scores(words: list, soup, u: str) -> dict:
        freq = Counter(words)
        total_tokens = len(words)

        h1, h2, h3, h4, h5, h6, anchor, title = _extract_headers_anchors_title_words(soup)
        url_host = _split_url_host(u)
        url_path = _split_url_path_and_query(u, url_host)

        headers_names = ["H1", "H2", "H3", "H4", "H5", "H6", "A", "Title", "URL-H", "URL-Q"]
        headers_scores = [6, 5, 4, 3, 2, 2, 1, 5, 5, 4]
        headers_lists = [h1, h2, h3, h4, h5, h6, anchor, title, url_host, url_path]

        word_info = {}  # word -> (freq, [tags], final_score)
        for w, c in freq.items():
            base = _tf_score(c, total_tokens)
            tag_boost, tag_names = 0.0, []
            for idx, toks in enumerate(headers_lists):
                if w in toks:
                    tag_boost += headers_scores[idx]
                    tag_names.append(headers_names[idx])
            word_info[w] = (c, tag_names, base + tag_boost)

        return word_info

    # --------------------- pipeline ---------------------
    clean_text, soup = _fetch_page(url)
    _, stopword_list = _detect_language_and_stopwords(clean_text)
    tokens = _clean_text_to_words(clean_text, stopword_list)

    if not tokens:
        return [] if not return_details else []

    keyword_data = _compute_keyword_scores(tokens, soup, url)
    top = sorted(keyword_data.items(), key=lambda kv: kv[1][2], reverse=True)[:k]

    if return_details:
        # [(word, freq, score, tags), ...]
        return [(w, meta[0], meta[2], meta[1]) for w, meta in top]
    else:
        # [word, ...]
        return [w for w, _ in top]


In [8]:
def get_top_keywords_from_html(html_content: str, url: str, k: int = 10, return_details: bool = False):
    """Process keywords from HTML content string instead of fetching from URL"""
    import re
    from urllib.parse import urlparse
    from collections import Counter
    from bs4 import BeautifulSoup, Comment
    from nltk.corpus import stopwords
    from nltk import wordpunct_tokenize
    import nltk
    
    # Ensure NLTK data
    try:
        stopwords.words("english")
    except LookupError:
        nltk.download("stopwords")
    
    # Config
    COMMON_NOISE_WORDS = set("""
    January debt est dec big than who use jun jan feb mar apr may jul august dec oct nov sep dec
    product continue one two three four five please thanks find helpful week job experience women girl
    apology read show eve knowledge benefit appointment street way staff salon discount gift cost thing
    world close party love letters rewards offers special close pack wed dollars voucher gifts vouchers
    welcome therefore march nights need name please show sisters thank menu today always time needs
    welcome march february april may june jully august september october november december day year
    month minute second seconds
    """.split())
    
    # Helper functions
    def _is_visible_text(element) -> bool:
        if element.parent.name in ["html", "style", "script", "head", "[document]", "img"]:
            return False
        if isinstance(element, Comment):
            return False
        return True
    
    def _extract_visible_text_from_html(html_str: str) -> str:
        soup = BeautifulSoup(html_str, "lxml")
        texts = soup.find_all(string=True)
        visible_texts = filter(_is_visible_text, texts)
        return " ".join(t.strip() for t in visible_texts)
    
    def _normalize_whitespace(text: str) -> str:
        lines = (line.strip() for line in text.splitlines())
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        return "\n".join(chunk for chunk in chunks if chunk)
    
    def _calculate_language_scores(text: str) -> dict:
        ratios = {}
        tokens = wordpunct_tokenize(text)
        words = [w.lower() for w in tokens]
        words_set = set(words)
        for lang in stopwords.fileids():
            try:
                stop_set = set(stopwords.words(lang))
            except Exception:
                continue
            common = words_set.intersection(stop_set)
            ratios[lang] = len(common)
        return ratios
    
    def _detect_language_and_stopwords(text: str):
        ratios = _calculate_language_scores(text)
        if not ratios:
            return "english", set(stopwords.words("english"))
        detected_lang = max(ratios, key=ratios.get)
        try:
            sw = set(stopwords.words(detected_lang))
        except Exception:
            detected_lang, sw = "english", set(stopwords.words("english"))
        return detected_lang, sw
    
    def _clean_text_to_words(text: str, stopword_list: set) -> list:
        words = []
        LETTERS_ONLY_RE = re.compile(r"[^a-zA-ZåäöÅÄÖ]+")
        for raw_word in text.split():
            token = LETTERS_ONLY_RE.sub("", raw_word).lower()
            if (len(token) > 1 and not token[0].isdigit() and token not in stopword_list 
                and token not in COMMON_NOISE_WORDS and not token.isdigit()):
                words.append(token)
        return words
    
    def _split_url_host(u: str) -> list:
        parsed = urlparse(u)
        host = parsed.hostname or ""
        parts = []
        for chunk in host.split("."):
            chunk = chunk.lower()
            if chunk not in ["", "https", "www", "com", "-", "php", "pk", "fi", "http"]:
                parts.append(chunk)
        return parts
    
    def _split_url_path_and_query(u: str, host_parts: list) -> list:
        path_tokens = []
        for segment in u.split("/"):
            for dot_part in segment.split("."):
                for dash_part in dot_part.split("-"):
                    token = dash_part.lower()
                    if (token and token not in ["https", "www", "com", "-", "php", "pk", "fi",
                        "https:", "http", "http:", "http:"] and token not in host_parts):
                        path_tokens.append(token)
        return path_tokens
    
    def _extract_tag_texts(soup, tag_name: str) -> list:
        out = []
        for el in soup.find_all(tag_name):
            t = el.get_text(strip=True).lower()
            if t:
                out.append(t)
        return out
    
    def _explode_texts_to_words(text_list: list) -> list:
        out = []
        for text in text_list:
            for comma_chunk in text.split(","):
                for w in comma_chunk.split():
                    out.append(w)
        return out
    
    def _extract_headers_anchors_title_words(soup):
        h1 = _explode_texts_to_words(_extract_tag_texts(soup, "h1"))
        h2 = _explode_texts_to_words(_extract_tag_texts(soup, "h2"))
        h3 = _explode_texts_to_words(_extract_tag_texts(soup, "h3"))
        h4 = _explode_texts_to_words(_extract_tag_texts(soup, "h4"))
        h5 = _explode_texts_to_words(_extract_tag_texts(soup, "h5"))
        h6 = _explode_texts_to_words(_extract_tag_texts(soup, "h6"))
        a = _explode_texts_to_words(_extract_tag_texts(soup, "a"))
        ti = _explode_texts_to_words(_extract_tag_texts(soup, "title"))
        return h1, h2, h3, h4, h5, h6, a, ti
    
    def _tf_score(freq: int, total_tokens: int) -> float:
        if total_tokens < 50:
            return (freq / 100.0) * 50
        else:
            return (freq / 100.0) * 20
    
    def _compute_keyword_scores(words: list, soup, u: str) -> dict:
        freq = Counter(words)
        total_tokens = len(words)
        h1, h2, h3, h4, h5, h6, anchor, title = _extract_headers_anchors_title_words(soup)
        url_host = _split_url_host(u)
        url_path = _split_url_path_and_query(u, url_host)
        
        headers_names = ["H1", "H2", "H3", "H4", "H5", "H6", "A", "Title", "URL-H", "URL-Q"]
        headers_scores = [6, 5, 4, 3, 2, 2, 1, 5, 5, 4]
        headers_lists = [h1, h2, h3, h4, h5, h6, anchor, title, url_host, url_path]
        
        word_info = {}
        for w, c in freq.items():
            base = _tf_score(c, total_tokens)
            tag_boost, tag_names = 0.0, []
            for idx, toks in enumerate(headers_lists):
                if w in toks:
                    tag_boost += headers_scores[idx]
                    tag_names.append(headers_names[idx])
            word_info[w] = (c, tag_names, base + tag_boost)
        return word_info
    
    # Process HTML content
    soup = BeautifulSoup(html_content, "lxml")
    clean_text = _extract_visible_text_from_html(html_content)
    clean_text = _normalize_whitespace(clean_text)
    
    _, stopword_list = _detect_language_and_stopwords(clean_text)
    tokens = _clean_text_to_words(clean_text, stopword_list)
    
    if not tokens:
        return [] if not return_details else []
    
    keyword_data = _compute_keyword_scores(tokens, soup, url)
    top = sorted(keyword_data.items(), key=lambda kv: kv[1][2], reverse=True)[:k]
    
    if return_details:
        return [(w, meta[0], meta[2], meta[1]) for w, meta in top]
    else:
        return [w for w, _ in top]


In [ ]:
# Get article folders (use first 300 or all available)
article_folders = get_article_folders(BASE, max_articles=300)
print(f"Found {len(article_folders)} articles to process")

if len(article_folders) == 0:
    print(f"No articles found. Checking path: {BASE}")
    print(f"Path exists: {os.path.exists(BASE)}")
    if os.path.exists(BASE):
        print(f"Contents: {os.listdir(BASE)[:5]}...")  # Show first 5 items
else:
    # Run evaluation with verbose output for debugging
    precision_sum, recall_sum, fscore_sum, total_webpages = Score_evaluation(
        article_folders, verbose=True
    )

    avg_p, avg_r, avg_f = Make_score_round_and_divide(
        precision_sum, recall_sum, fscore_sum, total_webpages
    )

    print("\n=== AVERAGE OVER ALL WEBPAGES ===")
    print("Average Precision:", avg_p)
    print("Average Recall   :", avg_r)
    print("Average F-score  :", avg_f)


Found 300 articles to process
Processing: 10/300 articles...
Processing: 20/300 articles...
Processing: 30/300 articles...
Processing: 40/300 articles...
Processing: 50/300 articles...
Processing: 60/300 articles...
Processing: 70/300 articles...
Processing: 80/300 articles...
Processing: 90/300 articles...
Processing: 100/300 articles...
Processing: 110/300 articles...
Processing: 120/300 articles...
Processing: 130/300 articles...
Processing: 140/300 articles...
Processing: 150/300 articles...
Processing: 160/300 articles...
Processing: 170/300 articles...
Processing: 180/300 articles...
Processing: 190/300 articles...
Processing: 200/300 articles...
Processing: 210/300 articles...
Processing: 220/300 articles...
Error processing article--india--india-others--beas-to-be-dried-up-in-search-for-bodies--: [Errno 2] No such file or directory: '/home/muditha/dataAnalytics/dataSets/indianexpress/article--india--india-others--beas-to-be-dried-up-in-search-for-bodies--/tags.txt'
Processing: 